In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os
import warnings

In [2]:
warnings.filterwarnings('ignore') # Hide warnings during grid search
print("--- Starting: Train & Tune Multi-Stage Career Recommender Model ---")

--- Starting: Train & Tune Multi-Stage Career Recommender Model ---


In [3]:
# --- 2. Load the Synthetic Dataset ---
try:
    # Corrected path based on your input
    dataset_path = '../Datasets/Career_Recommendation/new_synthetic_data/synthetic_career_data_multistage.csv'
    df = pd.read_csv(dataset_path)
    print(f"Synthetic dataset loaded successfully from: {dataset_path}")
except FileNotFoundError:
    print(f"Error: Dataset not found at '{dataset_path}'. Please ensure the file exists.")
    exit()

✅ Synthetic dataset loaded successfully from: ../Datasets/Career_Recommendation/new_synthetic_data/synthetic_career_data_multistage.csv


In [4]:
all_question_names = []
CAREER_QUESTIONS_LOCAL = { # Include the questions here for clarity
    'school': [{'name': 'logical'}, {'name': 'creativity'}, {'name': 'communication'}, {'name': 'curiosity'}, {'name': 'leadership'}, {'name': 'helping'}, {'name': 'technology'}, {'name': 'science'}, {'name': 'commerce'}, {'name': 'artistic'}],
    'intermediate': [{'name': 'math'}, {'name': 'physics'}, {'name': 'chemistry'}, {'name': 'biology'}, {'name': 'cs'}, {'name': 'economics'}, {'name': 'psychology'}, {'name': 'problem_solving'}, {'name': 'creativity'}, {'name': 'presentation'}],
    'btech': [{'name': 'programming'}, {'name': 'problem_solving'}, {'name': 'ai_ml'}, {'name': 'web_dev'}, {'name': 'security'}, {'name': 'core_eng'}, {'name': 'research'}, {'name': 'communication'}, {'name': 'teamwork'}, {'name': 'entrepreneurship'}]
}
for stage_questions in CAREER_QUESTIONS_LOCAL.values():
    all_question_names.extend([q['name'] for q in stage_questions])
all_question_names = sorted(list(set(all_question_names)))

In [5]:
feature_cols = ['stage'] + all_question_names
target_col = 'career_domain'

In [6]:
for col in feature_cols:
    if col not in df.columns:
        df[col] = 0

X = df[feature_cols]
y = df[target_col]

print(f"Features identified: {len(feature_cols)}")
print(f"Target identified: {target_col}")

Features identified: 28
Target identified: career_domain


In [7]:
print("\n--- Preprocessing Data (Encoding) ---")

# Target Encoding
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Target classes encoded: {le.classes_}")

# Feature Encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('stage_encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['stage'])
    ],
    remainder='passthrough'
)

X_processed = preprocessor.fit_transform(X)
print(f"\nTotal features after encoding: {X_processed.shape[1]}")


--- Preprocessing Data (Encoding) ---
Target classes encoded: ['Arts & Design' 'Commerce' 'Computer Science' 'Cybersecurity'
 'Data Science/AI' 'Engineering' 'Humanities' 'Management' 'Medical'
 'Product Management' 'Research' 'STEM' 'Software Development']

Total features after encoding: 30


In [8]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)
print(f"\nData split: {len(X_train)} training samples, {len(X_test)} testing samples.")

# --- 6. Hyperparameter Tuning using GridSearchCV ---
print("\n--- Tuning Hyperparameters with GridSearchCV ---")

param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'class_weight': ['balanced']
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1,
    scoring='accuracy'
)

print("Searching for best parameters...")
grid_search.fit(X_train, y_train)

print("\nBest parameters found:")
print(grid_search.best_params_)

best_rf_model = grid_search.best_estimator_
print("Hyperparameter tuning complete.")


Data split: 3840 training samples, 960 testing samples.

--- Tuning Hyperparameters with GridSearchCV ---
Searching for best parameters...
Fitting 5 folds for each of 36 candidates, totalling 180 fits

Best parameters found:
{'class_weight': 'balanced', 'max_depth': 20, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
✅ Hyperparameter tuning complete.


In [9]:
# --- 7. Model Evaluation (Using the Best Model) ---
print("\n--- Evaluating Tuned Model Performance ---")
y_pred = best_rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Tuned Model Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report (Tuned Model):")
print(classification_report(y_test, y_pred, labels=np.unique(y_encoded), target_names=le.classes_, zero_division=0))

# The rest of your script (saving artifacts, etc.) remains unchanged.


--- Evaluating Tuned Model Performance ---
🚀 Tuned Model Accuracy: 91.67%

Classification Report (Tuned Model):
                      precision    recall  f1-score   support

       Arts & Design       0.81      0.95      0.87        40
            Commerce       0.86      0.55      0.67        11
    Computer Science       0.80      0.87      0.83        23
       Cybersecurity       0.00      0.00      0.00         0
     Data Science/AI       0.95      0.88      0.92       119
         Engineering       0.96      0.96      0.96       248
          Humanities       0.91      0.89      0.90       151
          Management       0.67      0.59      0.62        17
             Medical       0.92      1.00      0.96        12
  Product Management       0.82      0.95      0.88        62
            Research       0.97      0.86      0.91        35
                STEM       0.94      0.93      0.93        99
Software Development       0.92      0.94      0.93       143

            accur

In [10]:
# --- 8. Save the Tuned Model and Encoders ---
print("\n--- Saving Tuned Model and Preprocessor ---")
output_dir = '../Models/Career_Recommendation' # Existing folder

base_filename = 'new_model_multistage_tuned' 
model_path = os.path.join(output_dir, f'{base_filename}.joblib')
preprocessor_path = os.path.join(output_dir, 'new_preprocessor.joblib')
label_encoder_path = os.path.join(output_dir, 'new_label_encoder.joblib')


--- Saving Tuned Model and Preprocessor ---


In [11]:
# Save the artifacts
joblib.dump(best_rf_model, model_path) # Save the best tuned model
joblib.dump(preprocessor, preprocessor_path)
joblib.dump(le, label_encoder_path)

['../Models/Career_Recommendation\\new_label_encoder.joblib']

In [12]:
print(f"Tuned model and encoders saved successfully in: {output_dir}")
print(f"   - Model: {os.path.basename(model_path)}")
print(f"   - Preprocessor: {os.path.basename(preprocessor_path)}")
print(f"   - Label Encoder: {os.path.basename(label_encoder_path)}")
print("--- Script Finished ---")

✅ Tuned model and encoders saved successfully in: ../Models/Career_Recommendation
   - Model: new_model_multistage_tuned.joblib
   - Preprocessor: new_preprocessor.joblib
   - Label Encoder: new_label_encoder.joblib
--- Script Finished ---
